# Phân tích Cấu trúc Doanh thu và Tối ưu hoá Tồn kho
## Ứng dụng Hybrid Time-Series Model và Prescriptive Analytics

**Datathon 2026 — Round 1 | Phần 2: Khám phá Dữ liệu & Đề xuất Vận hành**

---

### Bối cảnh

Tổng quan bài toán
Dữ liệu ghi nhận hành trình 10 năm (2012–2022) của một doanh nghiệp thương mại điện tử thời trang tại Việt Nam. Sau giai đoạn tăng trưởng nóng và đạt đỉnh vào năm 2016, doanh nghiệp hiện đang đối mặt với sự sụt giảm doanh thu mang tính hệ thống.

Phân tích này không chỉ nhìn lại quá khứ mà còn kết nối **6 nguồn dữ liệu cốt lõi** để xây dựng mô hình dự báo cho *18 tháng tới*, đồng thời đưa ra các đề xuất cụ thể nhằm tối ưu hóa hiệu quả vận hành.

### Bức tranh tổng thể từ các nguồn dữ liệu:
Việc kết nối các bảng dữ liệu giúp chúng ta hiểu rõ mối quan hệ nhân quả trong kinh doanh:

| Luồng Dữ liệu (Data Flow) | Mục tiêu Phân tích (Business Objective) |
| :--- | :--- |
| **`sales.csv` ↔ `inventory.csv`** | Loại bỏ "nhiễu" do tình trạng hết hàng gây ra để xác định nhu cầu thực tế. |
| **`sales.csv` ↔ `promotions.csv`** | Đánh giá mức độ hiệu quả của các chương trình giảm giá đến hành vi mua hàng. |
| **`sales.csv` ↔ `web_traffic.csv`** | Sử dụng dữ liệu truy cập như một chỉ báo sớm để dự đoán biến động doanh số. |
| **`products` → `order_items` → `orders`** | Phân tích hiệu suất theo từng danh mục hàng hóa. |
| **`Revenue` ↔ `COGS`** | Kiểm tra tính ổn định dài hạn giữa doanh thu và chi phí để tối ưu biên lợi nhuận. |

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

CSV_DIR = os.path.join(ROOT, "csv")
OUT_DIR = os.path.join(ROOT, "outputs", "part2")
os.makedirs(OUT_DIR, exist_ok=True)

# Load core data
sales = pd.read_csv(os.path.join(CSV_DIR, "sales.csv"), parse_dates=["Date"]).set_index("Date")
print(f"Sales data: {sales.index.min().date()} -> {sales.index.max().date()} ({len(sales)} rows)")
print(f"Revenue: mean={sales['Revenue'].mean()/1e6:.2f}M, std={sales['Revenue'].std()/1e6:.2f}M")
print(f"COGS:    mean={sales['COGS'].mean()/1e6:.2f}M, std={sales['COGS'].std()/1e6:.2f}M")
print(f"Corr(Revenue, COGS) = {sales[['Revenue','COGS']].corr().iloc[0,1]:.4f}")

### Nhận xét Tổng quan Dữ liệu Bán hàng (2012 - 2022)

Dựa trên tập dữ liệu lịch sử kéo dài hơn 10 năm (3833 quan sát theo ngày), các đặc trưng thống kê cốt lõi cho thấy:

*   **Tính biến động cao (High Volatility):** Doanh thu trung bình đạt 4.29M, nhưng độ lệch chuẩn (Standard Deviation) lên tới 2.62M. Sự phân tán dữ liệu lớn cho thấy nhịp độ bán hàng rất thiếu ổn định, phản ánh đặc thù của ngành thời trang với các rủi ro biến động theo mùa vụ và sự kiện.
*   **Biên lợi nhuận mỏng (Thin Margin):** Từ Revenue và COGS, ta có thể suy ra được Margin: Chênh lệch giữa doanh thu và giá vốn trung bình chỉ đạt 0.59M/ngày (tương đương biên lợi nhuận gộp khoảng 13.75%). Mức đệm lợi nhuận này đòi hỏi hệ thống vận hành phải cực kỳ tinh gọn, vì mọi sai số trong dự báo tồn kho đều có thể trực tiếp ăn mòn lợi nhuận ròng.
*   **Hệ số tương quan cực mạnh (Correlation = 0.9760):** Doanh thu và Giá vốn hàng bán (COGS) có mối quan hệ tuyến tính gần như tuyệt đối. 
    *   **Đánh giá nghiệp vụ:** Con số này chứng tỏ cơ cấu định giá của doanh nghiệp được duy trì rất kỷ luật trong suốt một thập kỷ. Ngay cả trong các đỉnh doanh thu (spike) do chạy chương trình giảm giá, tỷ lệ chi phí vốn trên doanh thu vẫn không bị phá vỡ. 

---
# 1. Phân tích Mô tả (Descriptive Analytics)

Mục tiêu: Đánh giá phân phối và xu hướng cấu trúc của doanh thu, chi phí trong giai đoạn 2012-2022.

## Viz 1: Doanh thu đạt đỉnh năm 2016 trước khi rơi vào chu kỳ suy giảm dài hạn

Biểu đồ mô tả **xu hướng đi xuống trong dài hạn**, với đường hồi quy tuyến tính ghi nhận độ giảm trung bình (slope) ở mức **-41M/năm**. 

**Vòng đời doanh thu phân hóa thành hai giai đoạn rõ rệt:**
*   **Giai đoạn 2012 – 2016 (Tăng trưởng nóng):** Doanh thu bứt phá mạnh mẽ từ mức nền 741M lên đỉnh 2,105M.
*   **Giai đoạn 2016 – 2022 (Suy giảm hệ thống):** Trượt dốc 44% từ đỉnh và bắt đầu đi ngang ở biên độ thấp, dao động trong khoảng 1,050M – 1,170M.

**Điểm nhấn phân tích (Key Finding):** 
Sức mua đang bị thu hẹp đáng kể về mặt cấu trúc. Tổng doanh thu năm 2022 (1,170M) không chỉ bốc hơi 44% so với đỉnh 2016 mà còn lùi sâu 29.4% so với giai đoạn đầu hoạt động (năm 2013 đạt 1,657M). 

**Khuyến nghị kỹ thuật (Prescriptive):** 
Mô hình Time-Series cần được cấu hình để nhận diện "điểm gãy cấu trúc" (Structural Break/Change Point) tại mốc năm 2019. Việc cắt bỏ hoặc giảm trọng số của dữ liệu thuộc chu kỳ tăng trưởng cũ (2012-2016) sẽ giúp thuật toán tránh bị nhiễu và bám sát chính xác động lực thị trường ở hiện tại.

In [ ]:
from src.analysis.descriptive import viz1_revenue_trend
path = viz1_revenue_trend(sales, OUT_DIR)
from IPython.display import Image, display
display(Image(filename=path))

## Viz 2: Cấu trúc Biên lợi nhuận gộp (Gross Margin) — Lỗ hổng tài chính từ cấu trúc Khuyến mãi

Biểu đồ Heatmap phơi bày sự phân hóa mạnh mẽ của tỷ suất sinh lời qua các quý. Điểm cốt lõi không nằm ở việc "có hay không chạy khuyến mãi", mà nằm ở **loại hình** và **tỷ lệ bao phủ (Coverage)** của chiến dịch:

*   **Quý 1 (Jan-Mar) đạt đỉnh 17.8%:** Dù vẫn triển khai các chương trình như *Spring Sale (12%)* hay *Rural Special (15%)* với tỷ lệ coverage đạt 34.5%, Q1 vẫn duy trì được biên lợi nhuận cao nhất năm. Lý do là các chiến dịch này sử dụng mô hình **Percentage Discount (Giảm theo %)** — có khả năng co giãn linh hoạt theo giá bán, đảm bảo lợi nhuận gộp không bao giờ trượt xuống mức âm.
*   **Quý 2 (Apr-Jun) đạt 17.0%:** Giai đoạn "gặt hái" tối ưu khi doanh nghiệp vừa hấp thụ được lượng cầu của mùa Hè, vừa bảo vệ được đệm lợi nhuận dày.
*   **Quý 4 (Oct-Dec) giảm xuống 10.8%:** Áp lực từ các chiến dịch xả hàng cuối năm (*Year-End Sale* giảm sâu ~20%) bắt đầu ăn mòn trực tiếp vào tỷ suất sinh lời.
*   **Quý 3 (Jul-Sep) chạm đáy 4.7%:** Đây là "vùng lõm" rủi ro nhất. Tỷ lệ coverage khuyến mãi vọt lên **68.8%** (gần gấp đôi Q1). Nghiêm trọng hơn, Q3 chịu ảnh hưởng nặng nề từ chiến dịch *Urban Blowout* (triển khai định kỳ 2 năm một lần). Việc áp dụng mô hình **Fixed Discount (Giảm cố định 50K/sản phẩm)** đã phá hủy hoàn toàn cấu trúc margin. Vì không có độ co giãn, các mặt hàng có giá bán thấp gần như bị bán lỗ dưới giá vốn, kéo biên độ tháng 8 rớt xuống mức âm (-8.3%).

**Khuyến nghị vận hành (Prescriptive):** 
Mô hình "Fixed Discount" chính là thủ phạm âm thầm bào mòn lợi nhuận. Doanh nghiệp cần thiết lập ngay cơ chế **kiểm soát biên độ sàn (Margin Guardrails)**. Khi áp dụng giảm giá cố định, hệ thống phải tự động vô hiệu hóa mã giảm giá nếu giá bán cuối cùng chạm ngưỡng Giá vốn hàng bán (COGS), nhằm bịt lại lỗ hổng tài chính trong Quý 3.

In [ ]:
from src.analysis.descriptive import viz2_profit_margin_heatmap
path = viz2_profit_margin_heatmap(sales, OUT_DIR)
display(Image(filename=path))

## Viz 3: Phân rã STL — Tính mùa vụ chi phối ~70% biến động doanh thu

Bằng việc áp dụng thuật toán **STL Decomposition** (Seasonal-Trend decomposition using Loess), chuỗi thời gian doanh thu được bóc tách thành ba thành phần cấu trúc độc lập:

*   **Xu hướng (Trend):** Khẳng định đà suy giảm mang tính hệ thống trong dài hạn, với tổng mức sụt giảm chạm ngưỡng 33.4% sau một thập kỷ (tốc độ suy giảm trung bình ~4%/năm).
*   **Mùa vụ (Seasonality):** Là động lực chi phối tuyệt đối, đóng góp tới **~70% tổng phương sai** của toàn bộ chuỗi dữ liệu.
*   **Sai số ngẫu nhiên (Residual):** Chiếm ~12% phần biến thiên còn lại, sinh ra từ các sự kiện cục bộ như tình trạng đứt gãy hàng hóa (stockout) hoặc các chiến dịch khuyến mãi chớp nhoáng.

**Phát hiện cốt lõi (Key Finding):** 
Sự áp đảo của yếu tố mùa vụ (mang trọng số lớn gấp 4 lần so với Trend) đã định hình rõ bản chất của tập dữ liệu. Đây chính là cơ sở toán học vững chắc để quyết định lựa chọn **Prophet** — thuật toán vốn được thiết kế tối ưu để bắt các chu kỳ phức tạp — làm tầng dự báo cơ sở (Base Model).

**Khuyến nghị vận hành (Prescriptive):** 
Doanh nghiệp cần thiết lập quy trình lập kế hoạch kinh doanh đặt trọng tâm **chủ động theo mùa vụ**. Đồng thời, ban quản trị cần trực diện nhìn nhận mức trượt dốc -4%/năm của xu hướng lõi và cần phân tích sâu hơn nguyên nhân gốc rễ của đà suy giảm trước khi đưa ra quyết định chiến lược.

In [ ]:
from src.analysis.descriptive import viz3_stl_decomposition
path = viz3_stl_decomposition(sales, OUT_DIR)
display(Image(filename=path))

## Viz 4: Phân rã Doanh thu theo Danh mục — Rủi ro Tập trung Doanh thu (Revenue Concentration Risk) và Sự dịch chuyển cơ cấu

**Luồng dữ liệu:** `products` → `order_items` → `orders`

Cấu trúc danh mục sản phẩm cho thấy sự mất cân đối sâu sắc, dòng tiền bị chi phối gần như hoàn toàn bởi một nhóm ngành hàng duy nhất:

| Danh mục (Category) | Doanh thu (M) | Tỷ trọng Doanh thu | Biên lợi nhuận (Gross Margin) | Đánh giá Nhóm hàng |
| :--- | :--- | :--- | :--- | :--- |
| **Streetwear** | 12,558 | **80.1%** | 9.3% | Core |
| **Outdoor** | 2,353 | 15.0% | 11.3% | Rising Star (Ngôi sao tiềm năng) |
| **Casual** | 440 | 2.8% | 7.7% | Laggard (Quy mô nhỏ, biên độ thấp) |
| **GenZ** | 329 | 2.1% | 15.5% | Niche (Ngách nhỏ, lãi dày) |

**Phát hiện cốt lõi (Key Findings):**
*   **Rủi ro mức độ tập trung (Concentration Risk):** Nhóm **Streetwear** là động lực chính mang về 80.1% dòng tiền với biên lợi nhuận bình quân 9.3%. Tuy nhiên, con số này che giấu sự phân cực cực đoan: margin khi không khuyến mãi đạt 19.7%, trong khi khi chạy khuyến mãi (đặc biệt Urban Blowout) margin rơi xuống -15.6%. Vấn đề không nằm ở bản thân sản phẩm, mà ở chính sách khuyến mãi. Bất kỳ sai số nào trong quản lý tồn kho (stockout) hoặc lạm dụng giảm giá ở nhóm hàng này sẽ lập tức đánh sập lợi nhuận của toàn doanh nghiệp.
*   **Đánh giá cấu trúc danh mục:** Nhóm **Outdoor** có độ lớn thị phần tương đối (15%) và biên lợi nhuận tốt (11.3%), cho thấy dư địa phát triển cực tốt. Đáng chú ý, nhóm **GenZ** tuy chỉ chiếm 2.1% tổng doanh thu nhưng lại sở hữu tỷ suất sinh lời vô địch (15.5%). Đối với **Casual**, đây là nhóm hàng có quy mô nhỏ (2.8%) và biên lợi nhuận thấp nhất (7.7%), cần đánh giá lại hiệu quả sử dụng vốn lưu động thay vì vội vàng kết luận là gánh nặng.

**Khuyến nghị chiến lược (Prescriptive):**
*   **Bảo vệ Dòng tiền (Defend the Core):** Ưu tiên tuyệt đối không gian kho và nguồn lực dự báo (Forecasting Engine) cho nhóm Streetwear. Phải đảm bảo tỷ lệ Service Level của nhóm này luôn ở mức cao nhất để không làm đứt gãy 80% doanh thu, đồng thời tái cấu trúc lại các chiến dịch khuyến mãi giảm giá cố định.
*   **Tối ưu Vốn lưu động (SKU Rationalization):** Đánh giá chi phí vận hành riêng biệt (chi phí lưu kho, xử lý đơn) của nhóm Casual trước khi đưa ra quyết định cắt giảm. Chuyển hướng tập trung phân bổ vốn và không gian lưu kho sang đẩy mạnh nhóm Outdoor và GenZ nhằm kéo tỷ suất sinh lời của toàn doanh nghiệp đi lên.

In [ ]:
from src.analysis.descriptive import viz4_revenue_by_category
path = viz4_revenue_by_category(
    os.path.join(CSV_DIR, "order_items.csv"),
    os.path.join(CSV_DIR, "products.csv"),
    os.path.join(CSV_DIR, "orders.csv"),
    OUT_DIR
)
display(Image(filename=path))

---
# 2. Phân tích Nguyên nhân (Diagnostic Analytics)

**Mục tiêu:** Bóc tách các triệu chứng bất thường trên biểu đồ (những cú spike tăng vọt hoặc những pha sụt giảm sâu) để xác định nguyên nhân gốc rễ (Root Cause) chi phối dòng tiền. 

Để giải quyết bài toán này, chúng ta không thể chỉ nhìn vào biến động tài chính tĩnh, mà bắt buộc phải áp dụng **Event Study** và **Cross-table joins** vì những lý do cốt lõi sau:

### 1. Tại sao phải sử dụng Event Study?
Biến động doanh thu hiếm khi là ngẫu nhiên; chúng thường là hệ quả của một tác động ngoại lai (can thiệp).
*   **Đo lường sự kiện (Event Quantification):** Kỹ thuật này cho phép thuật toán cô lập đường doanh thu tự nhiên (baseline) khỏi các cú sốc. Nó giúp chúng ta đo lường chính xác xem một sự kiện cụ thể (ví dụ: đợt giảm giá *Urban Blowout*) đã "uốn cong" đường xu hướng hay phá hủy biên lợi nhuận ở mức độ định lượng là bao nhiêu.
*   **Tránh bẫy nhân quả giả (Spurious Causality):** Giúp doanh nghiệp ngừng suy đoán cảm tính (ví dụ: "doanh thu tháng 8 rớt chắc do tháng cô hồn") để nhìn vào sự thật dữ liệu (doanh thu rớt vì cấu trúc Fixed Discount không phù hợp).

### 2. Tại sao bắt buộc phải dùng Cross-table joins (Kết nối bảng chéo)?
Dữ liệu đơn lẻ không thể kể một câu chuyện hoàn chỉnh. 
*   **Khớp nối Nguyên nhân và Kết quả:** Bảng `sales` (Doanh thu) chỉ cho bạn biết **Kết quả** (Triệu chứng). Để tìm ra **Nguyên nhân** (Căn bệnh), hệ thống phải join (ghép nối) nó với dữ liệu vận hành.
*   **Ví dụ thực tiễn:** 
    *   Nếu chỉ nhìn bảng `sales`, bạn thấy doanh thu Q3 rớt thảm hại. Nhưng khi **join với bảng `inventory`**, bạn mới phát hiện ra lý do thực sự là do đứt gãy chuỗi cung ứng (Stockout) — khách muốn mua nhưng không có hàng để bán. 
    *   Tương tự, việc **join `sales` với `promotions`** là cách duy nhất để đối chiếu dòng tiền thu về với các chiến dịch giảm giá, từ đó phát hiện ra "thủ phạm giết margin" như ở Viz 2.

## Viz 5: Khử nhiễu Stockout và Outlier — Khôi phục tín hiệu nhu cầu

**Kết nối dữ liệu:** `sales.csv` ↔ `inventory.csv` (stockout_flag)

Khi đánh giá hiệu suất bán hàng, một cái bẫy phân tích rất dễ gặp phải là nhầm lẫn giữa sự sụt giảm doanh thu do nhu cầu khách hàng yếu đi, với sự sụt giảm do thiếu hụt nguồn cung (stockout), hoặc bị nhiễu bởi các đỉnh doanh thu đột biến mang tính ngẫu nhiên.

**Phát hiện chính:** Dữ liệu tồn kho cho thấy có tới **67.3% tổ hợp sản phẩm-tháng (product-month)** ghi nhận ít nhất 1 ngày hết hàng. Tuy nhiên, tình trạng này thực tế chỉ xảy ra rải rác ở từng SKU riêng lẻ với mức trung bình 1.2 ngày/tháng (tương đương tỷ lệ theo product-day chỉ ở mức 3.8%). Mặc dù stockout gây ra hiện tượng **Méo mó tín hiệu nhu cầu (Demand Signal Distortion)**, nhưng do đặc thù mapping dữ liệu tồn kho từ cấp độ tháng xuống ngày gặp độ trễ, tác động thực sự của nó lên tổng thể là tương đối nhỏ. Điểm mù thực sự đe dọa đến độ chính xác của thuật toán máy học lại nằm ở các điểm dị biệt (outliers).

**Giải pháp Denoising:**
Để chuỗi thời gian phản ánh chính xác sức mua thực tế, quy trình làm sạch dữ liệu (Denoising) được thực hiện như sau:
1.  **Nội suy Stockout (Imputation):** Định vị các ngày thiếu hàng qua `stockout_flag` và thay thế doanh thu tại các điểm này bằng **trung bình trượt 7 ngày** (7-day rolling mean) để nối liền đường cong nhu cầu.
2.  **Kiểm soát điểm dị biệt (Outlier Capping):** Đây là thao tác khử nhiễu chủ đạo của hệ thống. Thuật toán tiến hành cắt ngọn (cap) các đỉnh doanh thu bất thường vượt quá dải băng an toàn (spike > mean + 3σ trong cửa sổ 30 ngày). Yếu tố sống còn ở bước này là hệ thống được thiết kế để **bảo tồn các đỉnh hệ thống (recurring spikes)** — tức các đỉnh doanh thu xuất hiện định kỳ ≥70% số năm vào cùng một mốc ngày/tháng.
**Kết quả:** Cung cấp một tập dữ liệu nền "sạch" và ổn định hơn. Mô hình học được chính xác quy luật mùa vụ thực tế mà không bị chệch hướng bởi các xung nhiễu ngẫu nhiên.

In [ ]:
# Denoise target first
from src.utils import load_sales, load_inventory_flags
from src.denoising import denoise_target

df = load_sales(os.path.join(CSV_DIR, "sales.csv"))
stockout_flags = load_inventory_flags(os.path.join(CSV_DIR, "inventory.csv"))
df = denoise_target(df, stockout_flags.get("stockout_flag", None))

from src.analysis.diagnostic import viz5_denoising_stockout
path = viz5_denoising_stockout(df, os.path.join(CSV_DIR, "inventory.csv"),
                                df["Clean_Revenue"], OUT_DIR)
display(Image(filename=path))

## Viz 6: Phân tích Sự kiện (Event Study) — Hiện tượng "Vay mượn nhu cầu" và Cú sốc Doanh thu 48 giờ

**Kết nối dữ liệu:** `sales.csv` ↔ `promotions.csv` (start_date, end_date)

**Phương pháp:** Áp dụng Event Study trên chiến dịch khuyến mãi — tính lift doanh thu trước/trong/sau mỗi đợt khuyến mãi (50 đợt).

**Phát hiện chính từ dữ liệu thực:**
- **Day 0:** Ngày đầu tiên ghi nhận Lift trung bình **+20.1%** (median +16.5%).
- **Day 1:** Hiệu ứng này tiếp tục với Lift duy trì **+17.5%**
- **Day 2:** Sang ngày 2, Lift tụt nhanh còn **+7.7%**
- **Day 3:** Lift **chuyển âm -12.7%** — doanh thu rơi DƯỚI mức bình thường
- **Day 4-5:** Nhu cầu khách hàng tiếp tục giảm, qua đó Lift tiếp tục âm (-19.6% đến -17.8%)
- IQR dao động từ 37% đến 55% → hiệu quả khuyến mãi không đồng nhất giữa các đợt

Hiệu quả phân hóa rõ theo loại chương trình: Fall Launch (giảm 10%) cho lift ổn định nhất, trong khi Year-End Sale (giảm 20%) thường xuyên gây lift âm ngay Day 0 do khách hàng đã quen chờ đợi và không bị kích thích mua sắm.

**Khuyến nghị:** Tổng doanh thu 14 ngày sau khi khởi chạy khuyến mãi trung bình giảm -3.6% (median -8.0%) so với kịch bản không khuyến mãi. Hiệu ứng **vay mượn nhu cầu (demand borrowing)** ở 48 giờ đầu không đủ bù đắp cú sụt kéo dài từ Day 3 trở đi.
1. Không lạm dụng khuyến mãi để kích cầu cấu trúc dài hạn
2. Nhu cầu ngắn hạn đòi hỏi lập kế hoạch tồn kho **tăng buffer trước khuyến mãi 3-5 ngày** để đón spike mà không bị stockout.

In [ ]:
from src.analysis.diagnostic import viz6_promotion_intervention
path = viz6_promotion_intervention(df, os.path.join(CSV_DIR, "promotions.csv"), OUT_DIR)
display(Image(filename=path))

## Viz 7: Web traffic và Doanh thu — Tương quan mùa vụ và Tín hiệu cảnh báo

**Kết nối dữ liệu:** `sales.csv` ↔ `web_traffic.csv` (date join)

**Phương pháp:** Sử dụng Hàm Tương quan chéo (Cross-Correlation Function - CCF) để đo lường độ trễ (lag) và chiều hướng tác động giữa sessions và Revenue.

**Phát hiện chính:**
- Tương quan giữa sessions và Revenue dao động quanh mức 0.32 ở mọi lag (0-7 ngày), không có đỉnh rõ rệt. Điều này cho thấy hai chuỗi có tương quan mùa vụ chung (co-movement) hơn là mối quan hệ dẫn trước-theo sau (leading indicator).
- Traffic ngày khuyến mãi **không tăng** (thậm chí giảm nhẹ 3.2%),
conversion rate ngày khuyến mãi giảm 9.6% so với ngày thường, cho thấy traffic trong các đợt promo có chất lượng thấp hơn (nhiều người vào xem nhưng không mua, hoặc mua với giá trị đơn hàng thấp hơn).

**Khuyến nghị:** Web traffic có thể dùng làm **hệ thống cảnh báo sớm**:
- Nếu sessions giảm >20% YoY liên tục 2-3 ngày → hệ thống sẽ phát cảnh báo khẩn cấp về nguy cơ lao dốc doanh thu ngay trong tuần đó.
- Tuy nhiên, tương quan dao động ở ngưỡng 32% - 37% không đủ sức mạnh phân tích nếu đứng độc lập → cần kết hợp nhiều tín hiệu

In [ ]:
from src.analysis.diagnostic import viz7_web_traffic_ccf
path = viz7_web_traffic_ccf(df, os.path.join(CSV_DIR, "web_traffic.csv"), OUT_DIR)
display(Image(filename=path))

## Viz 8: Revenue & COGS đồng liên kết — Phát hiện chi phí dội bất thường

**Phương pháp:** Sử dụng kiểm định Engle-Granger Cointegration Test nhằm xác định và đánh giá trạng thái cân bằng cốt lõi trong dài hạn giữa Doanh thu (Revenue) và Giá vốn hàng bán (COGS).

**Phát hiện chính:** Revenue và COGS **đồng liên kết** (p-value ≈ 0),
với hệ số cân bằng dài hạn **beta = 0.825** (cứ 1 đồng doanh thu tốn 0.825 đồng vốn). Hệ số beta = 0.825 được ước lượng bằng **hồi quy OLS** trên toàn bộ chuỗi 3,833 ngày (R² = 0.953), phản ánh cấu trúc chi phí trung bình dài hạn.

Khi chuỗi spread = COGS - (157,059 + 0.825 × Revenue) vượt quá +2 sigma,
doanh nghiệp đang bán hàng với chi phí bị dội lên bất thường.

**Từ dữ liệu thực:**
- **180 ngày** ghi nhận spread > +2 sigma (chi phí dội) — tập trung vào các đợt Urban Blowout
- **0 ngày** ghi nhận spread < -2 sigma — không tồn tại trường hợp tối ưu được biên lợi nhuận cao vượt trội

**Khuyến nghị:** Dùng chỉ số spread làm công cụ rà soát bất thường (anomaly detection)
đối với biên lợi nhuận. Tự động kích hoạt cảnh báo đỏ ngay khi bộ phận Marketing chạy chương trình
giảm giá cố định (fixed discount) có nguy cơ phá vỡ cấu trúc chi phí. Bất kỳ kịch bản định giá nào có biểu hiện nới rộng Spread vượt ngưỡng rủi ro (+2 Sigma) đều phải được yêu cầu tái thẩm định để bảo vệ cấu trúc chi phí lõi.

In [ ]:
from src.analysis.diagnostic import viz8_cointegration_anomaly
path = viz8_cointegration_anomaly(df, OUT_DIR)
display(Image(filename=path))

---
# 3. Mô hình Dự báo (Predictive Modeling)

Mục tiêu: Dự báo doanh thu 18 tháng (01/2023 – 07/2024) bằng pipeline cap43ra — kết hợp Prophet + LightGBM.

## Viz 9: Đánh giá Hiệu suất Mô hình — Sức mạnh đột phá của Kiến trúc Hybrid "cap43ra"

**So sánh 3 mô hình trên tập dữ liệu kiểm thử 2021-2022 (in-sample):**

| Mô hình | MAE | RMSE | R² |
|---------|-----|------|-----|
| Baseline (YoY Naive) | 0.774M | 1.092M | 0.5710 |
| Prophet Only | 0.863M | 1.128M | 0.5425 |
| **cap43ra (Hybrid)** | **0.551M** | **0.754M** | **0.7956** |

**So sánh 3 mô hình trên tập dữ liệu kiểm thử 2022 (out-of-sample)**

| Mô hình | MAE | RMSE | R² |
|---------|-----|------|-----|
| Baseline (YoY Naive) | 0.838M | 1.162M | 0.5182 |
| Prophet Only | 1.001M | 1.244M | 0.4479 |
| **cap43ra (Hybrid)** | **0.721M** | **0.954M** | **0.6754** |

*Lưu ý: Bảng In-sample chỉ dùng để tham khảo nhằm kiểm tra khả năng khớp dữ liệu của mô hình, con số R²=0.795 thường mang tính chất Overfitting. Bảng Out-of-sample mới là thước đo thực chiến. Nhìn vào bảng OOS, ta thấy thuật toán Prophet độc lập (R²=0.448) thậm chí còn thua xa phương pháp copy số liệu năm ngoái (Baseline R²=0.518). Tuy nhiên, khi kết hợp cùng LightGBM (Hybrid), mô hình lập tức đè bẹp Baseline và vươn lên mốc R²=0.675.*

**Tại sao cap43ra vượt trội?**
- Prophet học cấu trúc dài hạn, bao gồm xu hướng dài hạn (Trend) và tính mùa vụ (Seasonality) - yếu tố vốn dĩ chiếm tới 70% tổng biến động của dữ liệu.
- LightGBM bù đắp phần dư phi tuyến tính mà Prophet bỏ sót bằng lag-365 và calendar features.
- Tất cả features được thiết kế khép kín, đảm bảo **100%** không chứa dữ liệu từ tương lai.

In [ ]:
# Build validation predictions
from src.prophet_model import fit_prophet, predict_prophet
from src.lgbm_model import fit_lgbm_residual, predict_lgbm_residual, make_future_safe_features
from src.postprocess import blend_forecasts

TRAIN_END  = "2021-12-31"
VAL_START  = "2022-01-01"
VAL_END    = "2022-12-31"
TEST_START = "2023-01-01"
TEST_END   = "2024-07-01"

train = df.loc[:TRAIN_END]
full_idx = pd.date_range(df.index.min(), TEST_END)

# Revenue pipeline
train_series = train["Clean_Revenue"].dropna()
prophet_model = fit_prophet(train_series, target_name="Revenue")
prophet_preds = predict_prophet(prophet_model, str(df.index.min().date()), TEST_END)

train_prophet = prophet_preds.loc[:TRAIN_END, "prophet_pred"]
residual_train = train["Revenue"].reindex(train_prophet.index) - train_prophet

full_residuals = pd.Series(np.nan, index=full_idx, name="residual")
full_residuals.update(residual_train)
full_trend = prophet_preds["prophet_trend"].reindex(full_idx)
X_full = make_future_safe_features(full_residuals, full_trend, full_idx)

X_train = X_full.loc[:TRAIN_END]
y_train = residual_train.reindex(X_train.index)
lgbm_model = fit_lgbm_residual(X_train, y_train, target_name="Revenue")

# Validation predictions
val_prophet = prophet_preds.loc[VAL_START:VAL_END, "prophet_pred"]
X_val = X_full.loc[VAL_START:VAL_END]
val_resid = predict_lgbm_residual(lgbm_model, X_val)
val_gridbreaker = blend_forecasts(val_prophet, val_resid)

actual_val = df.loc[VAL_START:VAL_END, "Revenue"].reindex(val_gridbreaker.index).dropna()
val_gridbreaker = val_gridbreaker.reindex(actual_val.index)
baseline = df["Revenue"].shift(365).loc[VAL_START:VAL_END].reindex(actual_val.index)

val_results = {
    "Baseline (YoY Naive)": baseline,
    "Prophet Only": val_prophet.reindex(actual_val.index),
    "cap43ra": val_gridbreaker
}

from src.analysis.predictive import viz9_model_comparison
path = viz9_model_comparison(df, val_results, OUT_DIR)
display(Image(filename=path))

## Viz 10: Dự báo 18 tháng qua khoảng tin cậy 95%

Biểu đồ này là **cơ sở trực quan để tính toán mức tồn kho an toàn (safety stock)**.
Dải băng xanh nhạt bao quanh đường dự báo thể hiện **khoảng tin cậy 95%** — doanh thu thật có 95%
khả năng nằm trong dải này.

**Định lượng rủi ro:**
- Mô hình đạt sai số trung bình (MAPE) ở mức 13.4% trên tập validation. Về mặt thống kê, để bao phủ được biên độ dao động, dải băng tin cậy 95% thực tế được mở rộng lên mức **±22%** so với điểm dự báo trung tâm (point forecast).
- Doanh nghiệp cần chuẩn bị tồn kho: Tồn kho = Dự báo điểm (forecast) × (1 + 0.22) để đảm bảo 97.5% không bị stockout.

**Khuyến nghị:** Một mô hình dự báo tốt không chỉ cho một con số,
mà phải cho **khoảng tin cậy** để doanh nghiệp ra quyết định có tính toán rủi ro.

In [ ]:
# Forecast with prediction intervals
test_prophet_full = prophet_preds.loc[TEST_START:TEST_END]
X_test = X_full.loc[TEST_START:TEST_END]
lgbm_resid_pred = predict_lgbm_residual(lgbm_model, X_test)
test_final = blend_forecasts(test_prophet_full["prophet_pred"], lgbm_resid_pred)

resid_std = (actual_val - val_gridbreaker).std()
forecast_df = pd.DataFrame({
    "forecast":  test_final,
    "lower_95":  (test_final - 1.96 * resid_std).clip(lower=0),
    "upper_95":  test_final + 1.96 * resid_std,
}, index=test_final.index)

from src.analysis.predictive import viz10_forecast_with_intervals
path = viz10_forecast_with_intervals(df, forecast_df, OUT_DIR)
display(Image(filename=path))

## Viz 11: SHAP — Lịch sử trễ 365 ngày là động lực dự báo mạnh nhất

**Phương pháp:** Ứng dụng thuật toán SHAP qua cấu trúc TreeExplainer cho LightGBM residual model, qua đó đo lường mức độ đóng góp độc lập của từng biến số (features) vào kết quả bù đắp sai số cuối cùng.

**Phát hiện chính:** `resid_lag365` (sai lệch cùng ngày năm ngoái) là biến quan trọng nhất,
cho thấy **resid_lag365 quan trọng nhất**. Dữ liệu chứng minh phần dư (residual) mà thuật toán Prophet bỏ sót hoàn toàn không phải nhiễu ngẫu nhiên, mà mang tính **chu kỳ lặp lại** — nếu Prophet dự báo thiếu ở ngày X năm ngoái, nó có **xu hướng lặp lại sai lầm tương tự ở ngày X năm nay. LightGBM học được pattern này để bù đắp**.

**Business implication từ SHAP:**
- Vì lag-365 là driver #1 → Khẳng định **kế hoạch năm** là cốt lõi (annual planning)
- `prophet_trend` là driver #2 → Xác nhận **xu hướng dài hạn** vẫn quan trọng, không thể bỏ qua
- Calendar features (month, dayofweek) → **kế hoạch theo mùa** cần tinh chỉnh theo mùa trong năm và tập tính tiêu dùng vào cuối tuần

**Đề xuất:** Xây dựng kế hoạch cung ứng cốt lõi theo chu kỳ năm (YoY), thay vì phản ứng thụ động theo từng tháng.
Chỉ dùng quỹ dự phòng linh hoạt (Safety Stock Buffer) ở mức ~15% để ứng phó biến động ngắn hạn.

In [ ]:
import shap
from src.lgbm_model import FEAT_COLS

explainer = shap.TreeExplainer(lgbm_model)
X_shap = X_val.dropna().head(2000)
shap_values = explainer.shap_values(X_shap)

from src.analysis.predictive import viz11_shap_upgraded
path = viz11_shap_upgraded(shap_values, FEAT_COLS, X_shap, OUT_DIR)
display(Image(filename=path))

---
# 4. Khuyến nghị Vận hành (Prescriptive Actions)

Mục tiêu: Đưa ra các tham số vận hành tối ưu dựa trên kết quả phân phối dự báo.

## Viz 12: Tối ưu tồn kho — Buffer +15% giảm rủi ro stockout đáng kể

**Bài toán đánh đổi (Trade-off):**
- Buffer 0%: Stockout risk ~39.6%, chi phí lưu kho ~14.1%.
- Buffer +15%: Stockout risk giảm xuống ~25%, chi phí lưu kho tăng lên 24.4%.
- Buffer +25%: Stockout risk chỉ còn ~19.4%, nhưng chi phí tăng 32.2%.

**Điểm tối ưu:** +15% buffer — **giảm 14.6% rủi ro stockout** với **chi phí chỉ tăng 10.3%**.

**Đề xuất cụ thể:**
- **Giữa tháng (ngày 1-24):** Buffer +10% (rủi ro thấp hơn)
- **Cuối tháng (ngày 25-31):** Buffer +20% (doanh thu cuối tháng cao hơn giữa tháng +50.5% nên rủi ro stockout cao hơn hẳn)

In [ ]:
forecast_errors = (actual_val - val_gridbreaker) / val_gridbreaker

from src.analysis.prescriptive import viz12_safety_stock_tradeoff
path = viz12_safety_stock_tradeoff(df, forecast_errors,
                                    os.path.join(CSV_DIR, "inventory.csv"), OUT_DIR)
display(Image(filename=path))

## Viz 13: Mức độ ưu tiên Marketing — Tối ưu hóa Tỷ suất Hoàn vốn (ROI) theo Mùa vụ

**Kết nối dữ liệu:** `sales.csv` ↔ `web_traffic.csv` (conversion efficiency)

**Cơ sở Định lượng:** Thuật toán phân bổ ngân sách được xác định qua Điểm hoàn vốn (ROI Score) = Biên lợi nhuận gộp (Margin) × Hiệu suất chuyển đổi trên mỗi phiên truy cập (VND/session). Trọng số này sau đó được chuẩn hóa qua hàm Softmax để đưa ra tỷ lệ đầu tư tối ưu:

| Quý | Margin | Efficiency | Mức ưu tiên |
|-----|--------|-----------|-------------|
| **Quý 1** | 17.8% | 197 VND/session | **Top (~43%)** |
| **Quý 2** | 17.0% | 193 VND/session | **Cao (~37%)** |
| Quý 4 | 11.7% | 175 VND/session | Trung bình (~14%) |
| Quý 3 | 4.7%  | 167 VND/session | Tối thiểu (~5%) |


**Phát hiện chính:** Quý 1 và Quý 2 là hai đầu tàu tài chính của doanh nghiệp, hội tụ đủ cả hai yếu tố: margin dày nhất và khả năng vắt kiệt traffic thành dòng tiền hiệu quả nhất. Quý 3 có ROI thấp nhất do bị ảnh hưởng bởi Urban Blowout kéo margin xuống gần 0%.

**Đề xuất:** Dồn sức marketing vào Quý 1-Quý 2 (margin cao + conversion tốt).
Quý 3 chỉ khóa mức chi tiêu ở ngưỡng tối thiểu (3.8%) để duy trì nhận diện thương hiệu. Quý 4 giữ mức trung bình.

In [ ]:
from src.analysis.prescriptive import viz13_marketing_allocation
path = viz13_marketing_allocation(df, os.path.join(CSV_DIR, "web_traffic.csv"), OUT_DIR)
display(Image(filename=path))

## Viz 14: Hệ thống cảnh báo sớm (Early Warning System)

**Kết nối dữ liệu:** `sales.csv` ↔ `web_traffic.csv` ↔ `inventory.csv`

**Phương pháp:** So sánh lưu lượng truy cập (web traffic) YoY (cùng kỳ năm ngoái)
để loại bỏ nhiễu mùa vụ và phát hiện sụt giảm bất thường.

**Hệ thống Traffic Light:**

| Mức cảnh báo | Tín hiệu kích hoạt (Traffic YoY) | Khuyến nghị Hành động (S&OP) |
| :--- | :--- | :--- |
| 🟢 **XANH (An toàn)** | Tăng trưởng > 0% | Duy trì kế hoạch cung ứng cơ sở. |
| 🟡 **VÀNG (Rủi ro)** | Suy giảm từ 10% – 20% | Nâng dự phòng (buffer) thêm 10%, cập nhật forecast hàng tuần. |
| 🔴 **ĐỎ (Khẩn cấp)** | Suy giảm > 20% | Kích hoạt buffer thêm 20%, đàm phán tức thời với NCC, cắt giảm các chiến dịch khuyến mãi không cốt lõi. |

**Định lượng rủi ro từ dữ liệu (Data-driven Caveats):**
*   **Ngưỡng Vàng:** Khi lượng truy cập giảm từ 10-20%, xác suất kéo theo sự sụt giảm doanh thu đồng thời ghi nhận ở mức **54.8%** (chỉ nhỉnh hơn xác suất tung đồng xu một chút, cho thấy sự do dự của thị trường).
*   **Ngưỡng Đỏ:** Ngay cả khi lượng truy cập sụt giảm mạnh >20%, mức độ tương quan kéo theo doanh thu giảm cũng chỉ đạt **37.1%** trên một ngày đơn lẻ. 
*   **Điểm mù của chỉ báo độc lập:** Vì mức độ tương quan tuyến tính của Web Traffic là tương đối yếu, hệ thống sẽ dễ sinh ra "báo động giả" (false alarms). Để ra quyết định chính xác, tham số Traffic bắt buộc phải được thiết kế thành một ma trận điều kiện, kết nối đồng thời với các tín hiệu vận hành khác (như tỷ lệ *Stockout* và mức độ xói mòn *Margin*).

**Giá trị:** Chuyển từ **reactive** (chờ đợi stockout rồi xử lý)
sang **proactive** (dự đoán và phòng ngừa trước 2-3 ngày).

In [ ]:
from src.analysis.prescriptive import viz14_early_warning_system
path = viz14_early_warning_system(df, os.path.join(CSV_DIR, "web_traffic.csv"),
                                   os.path.join(CSV_DIR, "inventory.csv"), OUT_DIR)
display(Image(filename=path))

---
# Kết luận

## Tóm tắt các phát hiện chính

| Cấp độ | Phát hiện | Giá trị kinh doanh |
|--------|-----------|-------------------|
| **Descriptive** | Doanh thu đạt đỉnh 2016 rồi suy giảm cấu trúc -44%; Mùa vụ chiếm 70% biến động | Mô hình cần nhận diện structural break |
| **Diagnostic** | Urban Blowout (giảm giá cứng 50K) gây margin âm T8; Promo chỉ hiệu quả 2 ngày rồi âm | Denoising + quản lý khuyến mãi |
| **Predictive** | cap43ra đạt R²=0.932 (in-sample), R² = 0.704 (out-of-sample 2022) | Dự báo 18 tháng tin cậy |
| **Prescriptive** | Buffer +15% giảm rủi ro stockout đáng kể | Tối ưu chi phí tồn kho |

## Kết nối dữ liệu đã sử dụng

6 bảng dữ liệu được kết nối:
- **sales ↔ inventory** → Stockout denoising
- **sales ↔ promotions** → Intervention analysis (Event Study)
- **sales ↔ web_traffic** → Co-movement signal & Early Warning
- **products ↔ order_items ↔ orders** → Category analysis
- **Revenue ↔ COGS** → Cointegration & margin anomaly detection

## Đề xuất hành động

1. **Tồn kho:** Áp dụng buffer +15% (cuối tháng: +20%) → giảm rủi ro stockout đáng kể
2. **Marketing:** Ưu tiên Quý 1-Quý 2 (margin 17-18% + conversion cao nhất); Quý 3 chỉ duy trì tối thiểu
3. **Vận hành:** Triển khai hệ thống cảnh báo sớm kết hợp web traffic YoY + stockout flag + margin
4. **Kế hoạch:** Xây dựng annual plan làm cốt lõi (lag-365 là SHAP driver #1), dự phòng ~15% cho biến động ngắn hạn